In [ ]:
from google.colab import drive
drive.mount('/content/')

In [1]:
# !pip install chromadb
# !pip install phoenix
!pip install langgraph langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.17
    Uninstalling langchain-core-1.2.17:
      Successfully uninstalled langchain-core-1.2.17


In [14]:
#importing the required libraries
import pandas as pd
import numpy as np
import json
import re
from typing import List, Dict, Any, Tuple, TypedDict
from openai import OpenAI
import time
from dotenv import load_dotenv
import openai
import os
from datasets import load_dataset
# import chromadb
from datetime import datetime, timedelta
import random
import sqlite3
from opentelemetry.trace.status import Status, StatusCode
from opentelemetry.trace import get_current_span
# from phoenix.otel import register
# import logger
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage

# ✅ Load environment variables
load_dotenv()
openai_api_key = os.getenv("OPEN_AI_KEY")
# phoenix_endpoint = os.getenv("PHOENIX_COLLECTOR_ENDPOINT")

In [ ]:
# Agent Template
def agent_template(state) -> Dict:
    """ An AI Agent
    Args:
        state(Dict): the state of the system
    Returns:
        dict: the response action which may alter the state of the system
    """
    print("---AI Agent Description---")
    logger.info("")
    prompt = ""
    response = ""
    return {}

In [4]:
class AgentState(TypedDict):
    """This is the shared memory of the agentic system"""
    messages: list
    next_step: str

In [16]:
import os
from google.colab import userdata

# Securely fetch the API key from Colab Secrets
try:
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OpenAI API Key not found. Please add 'OPENAI_API_KEY' to your Colab Secrets (key icon on the left).")

llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)

In [20]:
# Settings for AI agents
def policy_expert(state: AgentState):
    """Expert1 checks polices of the insurance"""
    query = state.messages["messages"][-1].content
    response = llm.invoke(f"You are a Policy Expert. Answer this strictly based on insurance rules: {query}")
    return {"messages": [response]}

def claims_expert(state: AgentState):
    """Expert2 checks the claims from the clients"""
    query = state["messages"][-1].content
    reponse = llm.invoke(f"You are a Claims Agent. Guide the user on how to file a claim for: {query}")
    return {"messages": [reponse]}

def supervisor(state: AgentState):
    """The boss who decides which agent to work and whether to end the process"""
    last_message = state["messages"][-1]
    system_prompt = (
        "You are a Supervisor. You have two workers: 'Policy_Expert' and 'Claims_Expert'. "
        "Read the conversation and decide who should act next. "
        "If the user's question is answered, return 'FINISH'. "
        "Return ONLY the word: 'Policy_Expert', 'Claims_Expert', or 'FINISH'."
    )
    response = llm.invoke([SystemMessage(content=system_prompt), last_message])
    decision = response.content.strip()
    if decision not in ["Policy_Expert", "Claims_Expert"]:
        decision = "FINISH"
    return {"next_step": decision}


In [21]:
# Logic and Data flow
workflow = StateGraph(AgentState)
## Add nodes: (label, function)
workflow.add_node("Supervisor", supervisor)
workflow.add_node("Claims_Expert", claims_expert)
workflow.add_node("Policy_Expert", policy_expert)

workflow.set_entry_point("Supervisor")

## Add edges
workflow.add_edge("Policy_Expert", "Supervisor")
workflow.add_edge("Claims_Expert", "Supervisor")

### Add conditional edge (from, if condition, to vertices)
workflow.add_conditional_edges(
    "Supervisor",
    lambda state: state["next_step"], {
        "Claims_Expert": "Claims_Expert",
        "Policy_Expert": "Policy_Expert",
        "FINISH": END,
    }
)

In [26]:
# Run the agent
app = workflow.compile()
print("--- Insurance Multi-Agent System Started ---")
user_input = "I crashed my car. I need to pay back money. What should I do?"
print(f"user: {user_input}")
inputs = {"messages": [HumanMessage(content=user_input)]}
for output in app.stream(inputs):
    for key, value in output.items():
        print(f"{key}: {value}")
    if "messages" in value:
        print(f"agent: {value['messages'][-1].content}")
print("--- Insurance Multi-Agent System Ended ---")

--- Insurance Multi-Agent System Started ---
user: I crashed my car. I need to pay back money. What should I do?
Supervisor: {'next_step': 'Claims_Expert'}
Claims_Expert: {'messages': [AIMessage(content="I'm here to help you through the process of filing a claim for your car crash. Here are the steps you should take:\n\n### 1. **Ensure Safety First**\n   - Make sure that everyone involved in the accident is safe. If there are injuries, call emergency services right away.\n\n### 2. **Document the Accident**\n   - Gather information about the accident:\n     - Take photos of the damage to all vehicles involved.\n     - Record the date, time, and location of the accident.\n     - Exchange information (names, insurance details, contact information) with the other party.\n     - Gather witness contacts if available.\n\n### 3. **Contact Your Insurance Company**\n   - **Call Your Insurer:** Notify your insurance company about the accident as soon as possible. Many insurers have a time limit f